# Day 45 — Solutions: End-to-End Modeling Project
Pipeline + CV/tuning, holdout evaluation, save artifacts, serve via FastAPI.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve, average_precision_score
import numpy as np, joblib, json

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xho, ytr, yho = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=2000))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
param_grid = {'lr__C': [0.1, 1.0, 10.0]}
search = GridSearchCV(pipe, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)
search.fit(Xtr, ytr)
proba = search.predict_proba(Xho)[:,1]
roc = roc_auc_score(yho, proba); ap = average_precision_score(yho, proba)
prec, rec, th = precision_recall_curve(yho, proba); th = np.r_[0.0, th]
f1s = 2*prec*rec/(prec+rec+1e-12); i = np.nanargmax(f1s)
th_star, f1_star = float(th[i]), float(f1s[i])
meta = {'threshold': th_star, 'params': search.best_params_, 'metric': {'roc_auc': float(roc), 'ap': float(ap)}}
joblib.dump(search, 'model_search.joblib'); json.dump(meta, open('model_meta.json','w'))
{'cv_best_params': search.best_params_, 'holdout_roc_auc': float(roc), 'holdout_ap': float(ap), 'best_threshold': th_star, 'best_f1': f1_star}

## Minimal FastAPI app (app.py)

In [ ]:
open('app.py','w').write('''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib, json, numpy as np

app = FastAPI()
search = joblib.load('model_search.joblib')
meta = json.load(open('model_meta.json'))

class Features(BaseModel):
    x: list

@app.post('/predict')
def predict(req: Features):
    try:
        X = np.array([req.x], dtype=float)
        proba = float(search.predict_proba(X)[:,1][0])
        label = int(proba >= meta['threshold'])
        return {'proba': proba, 'label': label, 'threshold': meta['threshold']}
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))
''')
'Wrote app.py'